# Demostración de inferencia (Colab)
Carga los pesos entrenados (enlace del README) y clasifica ventanas de un archivo CWRU de principio a fin. No entrena nada.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/jafcn09/examenfinal_redes_uniingenieria.git"
WEIGHTS_URL = "<enlace a CNN1D_diffusion_s0.pt en Drive o Hugging Face>"
if not Path("src/cwru_dl").exists():
    if Path("../src/cwru_dl").exists():
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "cwru-bearing-diffusion"], check=True); os.chdir("cwru-bearing-diffusion")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, "src")

In [ ]:
import torch
from cwru_dl import CLASSES, load_config
from cwru_dl.data import download_cwru, load_signal, make_windows, zscore
from cwru_dl.models import CNN1D
from cwru_dl.train import weight_path

cfg = load_config("configs/full.yaml", profile="full")
download_cwru(cfg.root)
wp = weight_path(cfg, "CNN1D", "diffusion", 0)
if not wp.exists():
    if WEIGHTS_URL.startswith("http"):
        subprocess.run(["pip", "install", "-q", "gdown"], check=True); subprocess.run(["gdown", "--fuzzy", "-O", str(wp), WEIGHTS_URL], check=True)
    else:
        raise FileNotFoundError(f"Coloque los pesos en {wp} o defina WEIGHTS_URL")
model = CNN1D(); model.load_state_dict(torch.load(wp, map_location="cpu")); model.eval()
print("pesos cargados:", wp)

In [ ]:
for cls, load in [("Normal", 3), ("IR014", 3), ("OR021", 3), ("B007", 3)]:
    x_raw, rpm, name = load_signal(cfg, cls, load)
    w = torch.from_numpy(zscore(make_windows(x_raw[:cfg.win * 8], cfg.win, cfg.win)))
    with torch.no_grad():
        probs = model(w).softmax(1)
    pred = [CLASSES[i] for i in probs.argmax(1).tolist()]
    print(f"{name:14s} real={cls:7s} rpm={rpm:.0f} | predicho por ventana: {pred} | confianza media {probs.max(1)[0].mean():.3f}")